# Embeddings de frases: el texto en contexto

**Sesión 6 · IA Intermedio · EstudIA 2026**

En la sesión 5, una red recurrente apenas alcanzó a la bolsa de palabras. Hoy usamos **transformers** que alguien
más entrenó: primero vemos que le dan a cada palabra un vector según su contexto; después convertimos frases enteras
en vectores y los usamos para clasificar, con y sin ejemplos, y para buscar por significado.

**El recorrido.**

| Parte | Qué hacemos |
|---|---|
| 1 | Una palabra, varios significados |
| 2 | Una frase, un vector |
| 3 | Los tuits de la sesión 5, otra vez |
| 4 | Clasificar sin ejemplos (zero-shot) |
| 5 | Un clasificador encima de los embeddings |
| 6 | ¿Y con pocos ejemplos? |
| 7 | Buscar por significado |

**Antes de empezar.** *Archivo › Guardar una copia en Drive*. No necesitas GPU; si tienes una, todo va más rápido.
Ejecuta la celda de preparación (▶ o `Shift + Enter`).

In [ ]:
# Preparación: librerías. Ejecútala una vez.
import os, re, time, unicodedata, urllib.request, warnings
warnings.filterwarnings("ignore")                     # sin avisos de scikit-learn cuando hay un solo ejemplo por clase
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s06")
ARCHIVOS_S05 = ARCHIVOS.replace("/s06", "/s05")      # los textos del asistente vienen de la sesión 5
TUITS = "https://huggingface.co/api/datasets/cardiffnlp/tweet_sentiment_multilingual/parquet/spanish/{}/0.parquet"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from transformers.utils import logging as registro
registro.set_verbosity_error()   # sin avisos largos al cargar los modelos

DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(max(1, os.cpu_count() or 1))
print("Listo.", "Con GPU." if DISPOSITIVO.type == "cuda" else "Sin GPU: funciona igual, un poco más despacio.")

In [ ]:
# @title Funciones auxiliares: descargar archivos, escribir números, partir textos y dibujar similitudes
def descargar(nombre, url=None):
    """Baja un archivo (de la carpeta pública del curso, o de `url`) solo la primera vez y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", os.path.basename(nombre))
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(url or f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

def tokens(texto):
    """Parte un texto en palabras y signos, en minúsculas: "No me gustó." -> ["no", "me", "gustó", "."]."""
    return re.findall(r"\w+|[^\w\s]", str(texto).lower())

def mapa_de_parecidos(matriz, etiquetas, titulo):
    """Dibuja una matriz de similitudes con su valor en cada casilla."""
    n = len(etiquetas)
    fig, eje = plt.subplots(figsize=(1.6 * n + 3, 0.7 * n + 1.2))
    eje.imshow(matriz, cmap="Purples", vmin=float(np.min(matriz)) - 0.05, vmax=1)
    for i in range(n):
        for j in range(n):
            eje.text(j, i, f"{matriz[i, j]:.2f}".replace(".", ","), ha="center", va="center",
                     color="white" if matriz[i, j] > np.min(matriz) + 0.6 * (1 - np.min(matriz)) else "black")
    corto = [e if len(e) <= 34 else e[:32] + "…" for e in etiquetas]
    eje.set_xticks(range(n), [str(i + 1) for i in range(n)])
    eje.set_yticks(range(n), [f"{i + 1}. {e}" for i, e in enumerate(corto)])
    eje.set_title(titulo); plt.tight_layout(); plt.show()

---
## 1. Una palabra, varios significados

En la sesión 5, cada palabra tenía **un solo vector**: en fastText, “banco” es el mismo vector en un parque y en una
oficina. Un transformer lee la frase completa y, gracias a la **atención**, cada palabra mira a las demás antes de
decidir su vector.

Usamos **BETO**, un modelo tipo BERT entrenado con textos en español (Cañete y otros, 2020). Sacamos el vector que le
da a “banco” en cuatro frases y comparamos esos vectores con la **similitud del coseno**: 1 si apuntan en la misma
dirección, menos mientras más distintos sean.

In [ ]:
tok_beto = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
beto = AutoModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased").to(DISPOSITIVO).eval()

@torch.no_grad()
def vector_de_palabra(frase, palabra):
    """El vector que BETO le da a `palabra` dentro de `frase` (el promedio de sus tokens, en la última capa)."""
    t = tok_beto(frase, return_tensors="pt", return_offsets_mapping=True)
    posiciones = t.pop("offset_mapping")[0].tolist()
    inicio = frase.index(palabra); fin = inicio + len(palabra)
    cuales = [i for i, (a, b) in enumerate(posiciones) if a < fin and b > inicio and b > a]
    salida = beto(**t.to(DISPOSITIVO)).last_hidden_state[0]
    return F.normalize(salida[cuales].mean(0), dim=0).cpu()

FRASES_BANCO = ['Me senté en un banco del parque a leer.', 'El banco de la plaza está recién pintado.', 'Fui al banco a pagar la factura.', 'El banco me negó el crédito.']
V = torch.stack([vector_de_palabra(f, "banco") for f in FRASES_BANCO])
similitud_banco = (V @ V.T).numpy()
mapa_de_parecidos(similitud_banco, FRASES_BANCO, "“banco” según BETO: similitud del coseno")

**Para pensar.**

- ¿Qué pares de “banco” quedaron más parecidos? ¿Coincide con lo que tú entiendes?
- Con fastText, todas las casillas de esta matriz valdrían 1. ¿Por qué?

---
## 2. Una frase, un vector

Para clasificar o buscar textos queremos **un vector por frase**. Usamos
[`intfloat/multilingual-e5-small`](https://huggingface.co/intfloat/multilingual-e5-small) (licencia MIT): un
transformer pequeño, entrenado con muchísimos pares de frases para que las que dicen lo mismo queden cerca.

Su ficha dice dos cosas que hay que respetar: cada texto empieza con `"query: "`, y el vector de la frase es el
**promedio** de los vectores de sus tokens. Eso hace esta función.

In [ ]:
tok_e5 = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-small")
e5 = AutoModel.from_pretrained("intfloat/multilingual-e5-small").to(DISPOSITIVO).eval()

@torch.no_grad()
def embeber(textos, lote=64):
    """Un vector de 384 números por texto: el promedio de los vectores de sus tokens, de largo 1."""
    vectores = []
    for i in range(0, len(textos), lote):
        t = tok_e5(["query: " + str(x) for x in textos[i:i + lote]], padding=True, truncation=True,
                   max_length=128, return_tensors="pt").to(DISPOSITIVO)
        salida = e5(**t).last_hidden_state                       # un vector por token
        mascara = t["attention_mask"].unsqueeze(-1).float()       # 1 en los tokens reales, 0 en el relleno
        promedio = (salida * mascara).sum(1) / mascara.sum(1)
        vectores.append(F.normalize(promedio, dim=-1).cpu())
    return torch.cat(vectores).numpy()

FRASES = ['¿Va a llover el fin de semana?', '¿Qué clima hará el sábado?', 'Pídeme una pizza a domicilio.', 'Quiero pedir comida a la casa.', 'Apaga las luces de la sala.', 'Deja la sala a oscuras.']
E = embeber(FRASES)
print("Cada frase es un vector de", E.shape[1], "números.")
mapa_de_parecidos(E @ E.T, FRASES, "Similitud del coseno entre frases (e5)")

**Para pensar.**

- Las frases 1 y 2 no comparten ninguna palabra. ¿Qué similitud tendrían en una bolsa de palabras?
- En e5, casi todas las similitudes quedan altas. ¿Qué importa más: el valor o cuál frase queda más cerca?

---
## 3. Los tuits de la sesión 5, otra vez

Los mismos tuits en español (negativo, neutral, positivo). Ahora cada tuit es su embedding de e5, congelado, y encima
entrenamos una regresión logística. La parte del transformer no se entrena: solo la capa de encima.

El parámetro `C` controla cuánto se ajusta la regresión a los ejemplos: con `C` pequeño es más prudente, con `C` grande
se ajusta más. Lo elegimos con la validación.

In [ ]:
NOMBRES = ["negativo", "neutral", "positivo"]
ent, val, prueba = (pd.read_parquet(descargar(f"tuits_{p}.parquet", TUITS.format(p)))
                    for p in ("train", "validation", "test"))
inicio = time.time()
X_ent, X_val, X_prueba = embeber(list(ent.text)), embeber(list(val.text)), embeber(list(prueba.text))
print(f"{miles(len(ent) + len(val) + len(prueba))} tuits convertidos en vectores en {time.time() - inicio:.0f} s.")

def elegir_C(X, y, Xv, yv, opciones=(0.1, 1, 10, 100)):
    """Entrena una regresión logística por cada C y se queda con la de mejor exactitud de validación."""
    mejor = max(opciones, key=lambda C: (LogisticRegression(max_iter=3000, C=C).fit(X, y).predict(Xv) == yv).mean())
    print("C elegido con la validación:", mejor)
    return LogisticRegression(max_iter=3000, C=mejor).fit(X, y)

clasificador_tuits = elegir_C(X_ent, ent.label, X_val, val.label)
EXACTITUD_TUITS = (clasificador_tuits.predict(X_prueba) == prueba.label).mean()
print("Exactitud en prueba, embeddings de e5 + regresión logística:", pct(EXACTITUD_TUITS))

In [ ]:
filas = [("Bolsa de palabras (sesión 5)", 0.5230),
         ("GRU, vectores al azar (sesión 5)", 0.4851),
         ("GRU, vectores de fastText (sesión 5)", 0.5356),
         ("Embeddings de e5 + regresión logística", EXACTITUD_TUITS)]
fig, eje = plt.subplots(figsize=(8, 2.8))
eje.barh(range(4), [f[1] for f in filas], color=["#B0B0B0", "#B0B0B0", "#000000", "#CB318E"])
for i, f in enumerate(filas):
    eje.text(f[1] + 0.01, i, pct(f[1]), va="center")
eje.axvline(1 / 3, color="#7FC200", ls="--", lw=1.5, label="Responder siempre lo mismo (33 %)")
eje.set_yticks(range(4), [f[0] for f in filas]); eje.invert_yaxis(); eje.set_xlim(0, 1)
eje.set_xlabel(f"Exactitud con los {miles(len(prueba))} tuits de prueba"); eje.legend(frameon=False, loc="lower right")
eje.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

### ¿Y la negación?

Los dos pares de la sesión 5: mismas palabras, sentido opuesto. La bolsa de palabras les daba la misma respuesta, y la
GRU les daba la misma clase a las dos frases de cada par.

In [ ]:
PARES = [('no fue malo, fue bueno', 'no fue bueno, fue malo'), ('me gustó, no me aburrí', 'me aburrí, no me gustó')]
for a, b in PARES:
    E_par = embeber([a, b])
    for texto, fila in zip((a, b), clasificador_tuits.predict_proba(E_par)):
        print(f"   {texto:28s} → " + " · ".join(f"{NOMBRES[k]} {pct(fila[k], 0)}" for k in range(3)))
    print(f"   Similitud entre las dos frases: {float(E_par[0] @ E_par[1]):.2f}".replace(".", ","))
    print()

**Para pensar.**

- ¿Cuántas de las cuatro frases acertó? ¿Por qué las dos frases de cada par siguen teniendo vectores tan parecidos?
- La parte que se entrenó es la misma regresión logística de la bolsa de palabras. ¿Qué cambió, entonces?

---
## 4. Clasificar sin ejemplos (zero-shot)

Volvemos a la competencia de la sesión 5: lo que la gente le pide a un asistente de voz, en 60 intenciones. Esta vez
no entrenamos nada: escribimos **una frase que describe cada intención**, la convertimos en vector, y a cada texto le
damos la intención cuya descripción quede más cerca.

En la sesión 3 hicimos zero-shot de otra forma: el modelo leía el texto junto con cada etiqueta, una vez por etiqueta.
Aquí cada texto pasa por el modelo una sola vez, y las 60 descripciones también.

In [ ]:
# @title Una descripción por intención
DESCRIPCIONES = {
    'alarma_consultar': 'preguntar qué alarmas hay puestas',
    'alarma_poner': 'poner una alarma',
    'alarma_quitar': 'quitar o cancelar una alarma',
    'calendario_agendar': 'agendar una cita o un recordatorio en el calendario',
    'calendario_consultar': 'preguntar qué hay en el calendario',
    'calendario_quitar': 'borrar una cita del calendario',
    'clima_consultar': 'preguntar por el clima o el tiempo que hará',
    'cocina_consultar': 'preguntar algo sobre cocina',
    'cocina_receta': 'pedir una receta de cocina',
    'correo_agregar_contacto': 'agregar un contacto al correo',
    'correo_consultar': 'revisar si hay correos nuevos',
    'correo_consultar_contacto': 'preguntar los datos de un contacto',
    'correo_enviar': 'enviar un correo electrónico',
    'domicilio_consultar': 'preguntar por un pedido a domicilio',
    'domicilio_pedir': 'pedir comida a domicilio',
    'enchufe_apagar': 'apagar un enchufe inteligente',
    'enchufe_encender': 'encender un enchufe inteligente',
    'general_chiste': 'pedir que cuente un chiste',
    'general_otro': 'charlar o decir algo raro al asistente',
    'general_saludo': 'saludar al asistente',
    'hogar_cafe': 'pedir que prepare un café',
    'hogar_limpieza': 'pedir que la aspiradora limpie la casa',
    'hora_consultar': 'preguntar la hora o la fecha',
    'hora_convertir': 'convertir la hora a otra zona horaria',
    'listas_consultar': 'preguntar qué hay en una lista',
    'listas_crear_o_agregar': 'crear una lista o agregarle algo',
    'listas_quitar': 'quitar algo de una lista',
    'luces_apagar': 'apagar las luces',
    'luces_atenuar': 'bajar la intensidad de las luces',
    'luces_aumentar': 'subir la intensidad de las luces',
    'luces_cambiar': 'cambiar el color de las luces',
    'luces_encender': 'encender las luces',
    'musica_ajustes': 'cambiar los ajustes de la música, como el modo aleatorio',
    'musica_consultar': 'preguntar qué canción está sonando',
    'musica_me_gusta': 'decir que me gusta una canción',
    'musica_no_me_gusta': 'decir que no me gusta una canción',
    'noticias_consultar': 'pedir las noticias',
    'pregunta_bolsa': 'preguntar por el precio de una acción en la bolsa',
    'pregunta_dato': 'preguntar un dato o un hecho',
    'pregunta_definicion': 'preguntar qué significa una palabra',
    'pregunta_matematicas': 'hacer una cuenta o una pregunta de matemáticas',
    'pregunta_moneda': 'preguntar el cambio de una moneda',
    'recomendar_eventos': 'pedir recomendaciones de eventos',
    'recomendar_lugares': 'pedir recomendaciones de lugares o restaurantes',
    'recomendar_peliculas': 'pedir recomendaciones de películas',
    'redes_consultar': 'preguntar qué hay en las redes sociales',
    'redes_publicar': 'publicar algo en redes sociales',
    'reproducir_audiolibro': 'poner un audiolibro',
    'reproducir_juego': 'jugar un juego',
    'reproducir_musica': 'poner música',
    'reproducir_podcast': 'poner un pódcast',
    'reproducir_radio': 'poner una emisora de radio',
    'transporte_consultar': 'preguntar por horarios de trenes o buses',
    'transporte_taxi': 'pedir un taxi',
    'transporte_tiquete': 'comprar un tiquete de tren o avión',
    'transporte_trafico': 'preguntar cómo está el tráfico',
    'volumen_bajar': 'bajar el volumen',
    'volumen_otro': 'otra cosa con el volumen',
    'volumen_silenciar': 'silenciar el volumen',
    'volumen_subir': 'subir el volumen',
}

In [ ]:
asistente = pd.read_csv(descargar("asistente_entrenamiento.csv", f"{ARCHIVOS_S05}/asistente_entrenamiento.csv"))
a_ent, a_prueba = train_test_split(asistente, test_size=2000, stratify=asistente.intencion, random_state=0)
INTENCIONES = sorted(DESCRIPCIONES)
inicio = time.time()
A_ent, A_prueba = embeber(list(a_ent.texto)), embeber(list(a_prueba.texto))
print(f"{miles(len(asistente))} textos convertidos en vectores en {time.time() - inicio:.0f} s. "
      f"Separamos {miles(len(a_prueba))} para medir.")

D = embeber([f"el usuario quiere {DESCRIPCIONES[c]}" for c in INTENCIONES])
zero_shot = np.array(INTENCIONES)[np.argmax(A_prueba @ D.T, axis=1)]
EXACTITUD_ZERO_SHOT = (zero_shot == a_prueba.intencion.to_numpy()).mean()
print("Zero-shot, sin un solo ejemplo:", pct(EXACTITUD_ZERO_SHOT), "· responder siempre lo mismo: 7 %")
for texto, real, dijo in list(zip(a_prueba.texto, a_prueba.intencion, zero_shot))[:8]:
    print(f"   {'✓' if real == dijo else '✗'} {texto[:45]:45s} → {dijo}" + ("" if real == dijo else f" (era {real})"))

**Para pensar.**

- ¿En qué se equivoca el zero-shot? ¿Mejorarías alguna descripción?

---
## 5. Un clasificador encima de los embeddings

Ahora sí usamos los ejemplos: una regresión logística sobre los embeddings, con todos los textos de entrenamiento, y
la comparamos con la bolsa de palabras entrenada con los mismos textos.

In [ ]:
bolsa = TfidfVectorizer(tokenizer=tokens, lowercase=False, token_pattern=None, ngram_range=(1, 2), sublinear_tf=True)
pred_bolsa = LogisticRegression(max_iter=3000, C=10).fit(bolsa.fit_transform(a_ent.texto), a_ent.intencion).predict(
    bolsa.transform(a_prueba.texto))
pred_e5 = LogisticRegression(max_iter=3000, C=10).fit(A_ent, a_ent.intencion).predict(A_prueba)
EXACTITUD_BOLSA_TODO = (pred_bolsa == a_prueba.intencion.to_numpy()).mean()
EXACTITUD_E5_TODO = (pred_e5 == a_prueba.intencion.to_numpy()).mean()
print(f"Con {miles(len(a_ent))} textos de entrenamiento:")
print("   Bolsa de palabras:", pct(EXACTITUD_BOLSA_TODO))
print("   Embeddings de e5 + regresión logística:", pct(EXACTITUD_E5_TODO))

**Para pensar.**

- Con tantos ejemplos, ¿cuánto ganan los embeddings? ¿Por qué crees que la bolsa de palabras aguanta tan bien en esta tarea?

---
## 6. ¿Y con pocos ejemplos?

En un proyecto real casi nunca hay cientos de ejemplos por clase. Repetimos la comparación dándole a cada modelo solo
*k* ejemplos por intención, para varios valores de *k* (tres sorteos distintos para cada uno).

In [ ]:
KS = [1, 2, 4, 8, 16, 32, 64]
curva = {"bolsa": [], "e5": []}
y_ent, y_prueba = a_ent.intencion.to_numpy(), a_prueba.intencion.to_numpy()
for k in KS:
    resultados = []
    for sorteo in range(3):
        rng = np.random.default_rng(sorteo)
        idx = np.concatenate([rng.permutation(np.where(y_ent == c)[0])[:k] for c in INTENCIONES])
        b = TfidfVectorizer(tokenizer=tokens, lowercase=False, token_pattern=None, ngram_range=(1, 2), sublinear_tf=True)
        p1 = LogisticRegression(max_iter=2000, C=10).fit(b.fit_transform(a_ent.texto.to_numpy()[idx]), y_ent[idx]).predict(
            b.transform(a_prueba.texto))
        p2 = LogisticRegression(max_iter=2000, C=10).fit(A_ent[idx], y_ent[idx]).predict(A_prueba)
        resultados.append(((p1 == y_prueba).mean(), (p2 == y_prueba).mean()))
    curva["bolsa"].append(float(np.mean([r[0] for r in resultados])))
    curva["e5"].append(float(np.mean([r[1] for r in resultados])))
    print(f"k = {k:2d}: bolsa de palabras {pct(curva['bolsa'][-1])} · embeddings {pct(curva['e5'][-1])}")

fig, eje = plt.subplots(figsize=(8, 3.4))
eje.plot(KS, curva["bolsa"], "o-", color="#B0B0B0", lw=2, label="Bolsa de palabras")
eje.plot(KS, curva["e5"], "o-", color="#CB318E", lw=2, label="Embeddings de e5 + regresión logística")
eje.axhline(EXACTITUD_ZERO_SHOT, color="#000000", ls="--", lw=1.2, label=f"Zero-shot, sin ejemplos ({pct(EXACTITUD_ZERO_SHOT)})")
eje.set_xscale("log", base=2); eje.set_xticks(KS, [str(k) for k in KS]); eje.set_ylim(0, 1)
eje.set_xlabel("Ejemplos por intención"); eje.set_ylabel("Exactitud en prueba"); eje.grid(alpha=0.3)
eje.legend(frameon=False, loc="lower right"); eje.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

**Para pensar.**

- ¿Con cuántos ejemplos por intención alcanza la bolsa de palabras al zero-shot?
- Tu proyecto tiene de 20 a 50 ejemplos. ¿En qué parte de la curva está?

---
## 7. Buscar por significado

Con los vectores también podemos **buscar**: convertimos una pregunta en vector y traemos los textos más cercanos. No
hace falta que compartan palabras. Esta es la mitad de un sistema RAG, que construiremos en la sesión 9.

In [ ]:
def buscar(pregunta, cuantos=4):
    """Los textos de entrenamiento más cercanos a la pregunta (sin repetir textos)."""
    q = embeber([pregunta])[0]
    print(f"“{pregunta}”")
    vistos = set()
    for i in np.argsort(-(A_ent @ q)):
        texto = a_ent.texto.to_numpy()[i]
        clave = unicodedata.normalize("NFKD", texto.lower()).encode("ascii", "ignore")   # sin tildes
        if clave in vistos:
            continue
        vistos.add(clave)
        print(f"   {float(A_ent[i] @ q):.2f}  {texto}".replace(".", ",", 1))
        if len(vistos) == cuantos:
            break

for pregunta in ['quiero escuchar algo de salsa', 'me muero de hambre', 'apaga todo que me voy a dormir']:
    buscar(pregunta)

**Para pensar.**

- ¿Alguno de los resultados comparte palabras con la pregunta? ¿Cuál resultado sobra?
- ¿Qué buscarías tú entre los textos de tu proyecto?

**Lo que viene.** Hoy el transformer quedó congelado y solo entrenamos la capa de encima. En la sesión 7 lo
**ajustamos**: entrenamos también el transformer con nuestros ejemplos, como hicimos con el modelo de imágenes en la
sesión 4.

**Fuentes.** Tuits: Barbieri, F., Espinosa Anke, L. y Camacho-Collados, J. (2022). XLM-T. *LREC*.
`cardiffnlp/tweet_sentiment_multilingual` (español), CC BY 3.0 y términos de Twitter, para uso académico. Textos del
asistente: FitzGerald, J. y otros (2023). MASSIVE. *ACL*. CC BY 4.0. BETO: Cañete, J. y otros (2020). Spanish
Pre-Trained BERT Model and Evaluation Data. *PML4DC, ICLR*. CC BY 4.0. e5: Wang, L. y otros (2024). Multilingual E5
Text Embeddings: A Technical Report. Licencia MIT.